#### Environment Check

In [ ]:
# Find the lab folder whether this notebook is run from notebooks/ or retrieval-lab/.
import sys
from pathlib import Path

PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name == "notebooks":
    LAB_DIR = PROJECT_DIR.parent
elif PROJECT_DIR.name == "retrieval-lab":
    LAB_DIR = PROJECT_DIR
else:
    LAB_DIR = PROJECT_DIR / "06-best-practices" / "retrieval-lab"

CODE_DIR = LAB_DIR / "code"

sys.path.append(str(CODE_DIR))

print("Python:", sys.executable)
print("Lab dir:", LAB_DIR)
print("Code dir:", CODE_DIR)

#### Load Search Helpers

In [ ]:
# Load the shared search helpers and create reusable clients.
from elasticsearch import ConnectionError

from search import (
    create_es_client,
    create_embedding_model,
    keyword_search,
    vector_search,
    hybrid_search,
)

es_client = create_es_client()

try:
    es_info = es_client.info()
except ConnectionError as error:
    raise RuntimeError(
        "Elasticsearch is not running. From retrieval-lab, run: "
        "docker compose up -d"
    ) from error

embedding_model = create_embedding_model()

es_info


#### Check Elasticsearch Index

In [ ]:
# This index must exist before search can work. If it is missing, run code/ingest.py.
INDEX_NAME = "course-questions"

index_exists = es_client.indices.exists(index=INDEX_NAME)

print("Index exists:", index_exists)

if not index_exists:
    raise RuntimeError(
        "The Elasticsearch index is missing. From retrieval-lab, run: "
        "uv run python code/ingest.py"
    )


#### Keyword Search

In [ ]:
# Keyword search is strongest when the user query shares exact words with the documents.
query = "I just discovered the course. Can I still join?"

keyword_results = keyword_search(
    query=query,
    num_results=5,
    es_client=es_client,
)

for doc in keyword_results:
    print(doc["course"], "-", doc["question"])

#### Vector Search

In [ ]:
# Vector search uses embeddings to find documents with similar meaning.
vector_results = vector_search(
    query=query,
    num_results=5,
    es_client=es_client,
    embedding_model=embedding_model,
)

for doc in vector_results:
    print(doc["course"], "-", doc["question"])

#### Hybrid Search

In [ ]:
# Hybrid search combines keyword matching and vector similarity.
hybrid_results = hybrid_search(
    query=query,
    num_results=5,
    es_client=es_client,
    embedding_model=embedding_model,
)

for doc in hybrid_results:
    print(doc["course"], "-", doc["question"])

#### Compare Results

In [ ]:
# Print each method in the same format so the results are easy to compare.
def show_results(title, results):
    print(title)
    print("-" * len(title))

    for i, doc in enumerate(results, start=1):
        print(i, doc["course"], "-", doc["question"])

    print()


show_results("Keyword Search", keyword_results)
show_results("Vector Search", vector_results)
show_results("Hybrid Search", hybrid_results)

#### Short Learning Summary

In [ ]:
print("""
Keyword search is good when the query uses exact words from the documents.

Vector search is good when the query has similar meaning but different words.

Hybrid search combines both approaches so the retriever has a better chance of finding useful context.
""")